# Build detailed Stats Tables of penalty situations
- takes in penalty_summary and scoring_summary fro DB

### Dependencies / DIR / DB Connect

In [15]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from collections import defaultdict
import sqlite3


# ======= BASE PATHS =======
try:
    # Works when running as a script
    base_dir = Path(__file__).resolve().parent
except NameError:
    # Fallback for notebooks or interactive mode
    base_dir = Path.cwd()

# one directory up (your config.py lives here)
config_folder = base_dir.parent

# two directories up (for TEMP, data, images)
project_root = base_dir.parent.parent



# ======= DATA FOLDERS =======
temp_folder = project_root / "TEMP"
data_folder = project_root / "data"
roster_folder = data_folder / "player_info"
school_info_folder = data_folder / "school_info"

# ======= IMAGE FOLDERS =======
img_folder = project_root / "images"
logo_folder = img_folder / "logos"
background_folder = img_folder / "background"
plot_folder = project_root / "TEMP" / "IMAGE" / "scatter_plots"

# ======= IMPORT CONFIG =======
# ====== DB Path settings Ect ======
sys.path.insert(0, str(config_folder))
import config  # now you can import config.py

# ======= LOAD DATA =======
roster_file = roster_folder / "roster_10_30_25.csv"
roster_df = pd.read_csv(roster_file)
roster_df["Current Team"] = roster_df["Current Team"].replace("RPI", "Rensselaer")
school_info_file = school_info_folder / "arena_school_info.csv"
school_info_df = pd.read_csv(school_info_file)

conn = sqlite3.connect(data_folder /config.recent_clean_db)
cursor = conn.cursor()
print("Connected to database:", config.recent_clean_db)
# Print table names to verify connection
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Tables in the database:")
for table in tables:
    print(table[0])

Connected to database: ../data/db/Season_YTD.db
Tables in the database:
game_details
scoring_summary
penalty_summary
goalie_stats
player_stats
line_chart
linescore
advanced_metrics
shot_events
player_stats_ytd


## New Starting Point - WED 2-18

In [16]:
import pandas as pd
import numpy as np
from collections import defaultdict

# ============================
# INPUTS (fresh copies)
# ============================
pen = pd.read_sql_query("SELECT * FROM penalty_summary", conn)
goals = pd.read_sql_query("SELECT * FROM scoring_summary", conn)


def tag_coincidental_by_timestamp(pen: pd.DataFrame) -> pd.DataFrame:
    """
    Mark penalties as coincidental ONLY when both teams have penalties
    at the exact same (Game_ID, Period, Time) timestamp.

    Behavior:
    - If 1 vs 1 at same timestamp -> both flagged coincidental (no manpower effect)
    - If 2 vs 1 at same timestamp -> 1 from each flagged coincidental, 1 remains manpower
    - Works for minors/majors as long as Pen_Length is in the table
    """
    pen = pen.copy()
    pen["is_coincidental"] = False

    # stable row id so we can flag specific rows
    pen["_rid"] = np.arange(len(pen))

    # group by exact timestamp (Game_ID + Period + Time)
    gcols = ["Game_ID", "Period", "Time"]
    for _, g in pen.groupby(gcols, sort=False):
        teams = g["Team"].dropna().unique().tolist()
        if len(teams) != 2:
            continue

        t1, t2 = teams[0], teams[1]
        idx1 = g[g["Team"] == t1]["_rid"].tolist()
        idx2 = g[g["Team"] == t2]["_rid"].tolist()

        k = min(len(idx1), len(idx2))
        if k <= 0:
            continue

        # Flag k penalties from each side as coincidental.
        # (Choice among multiple at same time isn't perfect, but it's consistent and correct for manpower.)
        pen.loc[pen["_rid"].isin(idx1[:k]), "is_coincidental"] = True
        pen.loc[pen["_rid"].isin(idx2[:k]), "is_coincidental"] = True

    return pen.drop(columns=["_rid"])

# --- apply it ---
pen = tag_coincidental_by_timestamp(pen)

# =============================
# PATCH FOR OFFSELTING PENALTIES BY TIMESTAMP (BUG FIX)
# ============================
# HELPERS
# ============================
def period_start_seconds(period: str) -> int:
    p = str(period).strip()
    return {
        "1st Period": 0,
        "2nd Period": 20 * 60,
        "3rd Period": 40 * 60,
        "Overtime":   60 * 60,
        "OT":         60 * 60,
    }.get(p, 0)

def parse_mmss(x) -> int:
    s = str(x).strip()
    if ":" not in s:
        try:
            return int(float(s))
        except Exception:
            return 0
    m, sec = s.split(":")
    return int(m) * 60 + int(sec)

def sit_label(team_skaters: int, opp_skaters: int, phase: str) -> str:
    # user explicitly wants 3v3 split by REG vs OT
    if team_skaters == 3 and opp_skaters == 3:
        return "3v3 (OT)" if phase == "OT" else "3v3 (NON OT)"
    return f"{team_skaters}v{opp_skaters}"
REG_BASE = 5
OT_BASE  = 3

def strengths(active, t, away, home):
    phase = "OT" if t >= 3600 else "REG"

    if phase == "OT":
        # keep your OT logic (3v3 baseline)
        cA = sum(1 for p in active if (p["team"] == away and not p["ended"] and not p.get("is_coincidental", False)))
        cH = sum(1 for p in active if (p["team"] == home and not p["ended"] and not p.get("is_coincidental", False)))
        sA = OT_BASE + max(0, cH - cA)
        sH = OT_BASE + max(0, cA - cH)
        return int(sA), int(sH), phase

    # REG: only penalties NOT tagged coincidental affect manpower
    cA = sum(1 for p in active if (p["team"] == away and not p["ended"] and not p.get("is_coincidental", False)))
    cH = sum(1 for p in active if (p["team"] == home and not p["ended"] and not p.get("is_coincidental", False)))

    sA = max(3, REG_BASE - cA)
    sH = max(3, REG_BASE - cH)
    return int(sA), int(sH), phase


# ============================
# PREP: absolute time columns
# ============================
# penalty table expected columns: Period, Team, Player, Pen_Length, Penalty_Type, Time, Game_ID
pen["Pen_Length"] = pd.to_numeric(pen["Pen_Length"], errors="coerce")
pen = pen[pen["Pen_Length"].notna()].copy()
pen["Pen_Length"] = pen["Pen_Length"].astype(int)

pen["t"] = pen["Time"].apply(parse_mmss) + pen["Period"].apply(period_start_seconds)
goals["t"] = goals["Time"].apply(parse_mmss) + goals["Period"].apply(period_start_seconds)

# Expand 4-min double minors into 2 + 2 (approximation)
expanded = []
for _, r in pen.iterrows():
    L = int(r["Pen_Length"])
    if L == 4:
        r1 = r.copy(); r2 = r.copy()
        r1["Pen_Length"] = 2
        r2["Pen_Length"] = 2
        r2["t"] = float(r["t"]) + 120.0
        expanded.extend([r1, r2])
    else:
        expanded.append(r)
pen = pd.DataFrame(expanded).reset_index(drop=True)

# Keep typical manpower penalties (2/5). (If you want 10-min misconducts to matter, that’s a different model.)
pen = pen[pen["Pen_Length"].isin([2, 5])].copy()

# Team mapping per game (preferred from goals)
if {"Away_Team", "Home_Team"}.issubset(goals.columns):
    game_teams = goals.groupby("Game_ID")[["Away_Team", "Home_Team"]].first().reset_index()
else:
    # fallback: infer from goals Team col
    gt = goals.groupby("Game_ID")["Team"].unique().reset_index()
    rows = []
    for _, r in gt.iterrows():
        teams = list(r["Team"])
        if len(teams) >= 2:
            rows.append({"Game_ID": r["Game_ID"], "Away_Team": teams[0], "Home_Team": teams[1]})
    game_teams = pd.DataFrame(rows)

# ============================
# ACCUMULATORS
# ============================
# key = (Team, SituationLabel)
SECONDS   = defaultdict(float)
GF        = defaultdict(int)
GA        = defaultdict(int)
INSTANCES = defaultdict(int)

def add_instance(team, label, last_label_by_team):
    # count a new "instance" when a team enters a label different from previous
    if last_label_by_team.get(team) != label:
        INSTANCES[(team, label)] += 1
        last_label_by_team[team] = label

# ============================
# GAME SIMULATION
# ============================
def simulate_game(game_id, away, home, ot_length=300):
    pgame = pen[pen["Game_ID"] == game_id].copy().sort_values("t").reset_index(drop=True)
    ggame = goals[goals["Game_ID"] == game_id].copy().sort_values("t").reset_index(drop=True)

    # penalty objects
    penalties = []
    for _, r in pgame.iterrows():
        penalties.append({
            "team": str(r["Team"]),
            "start": float(r["t"]),
            "length": int(r["Pen_Length"]),
            "is_major": int(r["Pen_Length"]) == 5,
            "end": float(r["t"]) + int(r["Pen_Length"]) * 60.0,
            "ended": False,
        })

    # events: penalty starts + goals (penalty starts processed first at same timestamp)
    events = []
    for i, r in pgame.iterrows():
        events.append(("P_START", float(r["t"]), i))
    for j, r in ggame.iterrows():
        events.append(("GOAL", float(r["t"]), j))
    events.sort(key=lambda x: (x[1], 0 if x[0] == "P_START" else 1))

    max_t = max([0.0] + pgame["t"].tolist() + ggame["t"].tolist())
    end_time = 3600.0 if max_t <= 3600.0 else 3600.0 + float(ot_length)

    active = []
    ptr = 0
    cur = 0.0

    # track state transitions for Instances
    last_label = {}  # per team, carries across phases as label already bakes OT vs NON-OT for 3v3

    while cur < end_time - 1e-9:
        next_event_t = events[ptr][1] if ptr < len(events) else end_time
        next_end_t = min([p["end"] for p in active if not p["ended"]] + [end_time])
        nxt = min(next_event_t, next_end_t, end_time)

        # allocate segment [cur, nxt)
        if nxt > cur:
            sA, sH, phase = strengths(active, cur, away, home)
            lab_away = sit_label(sA, sH, phase)
            lab_home = sit_label(sH, sA, phase)

            # instances: entering a new label
            add_instance(away, lab_away, last_label)
            add_instance(home, lab_home, last_label)

            # time
            dt = nxt - cur
            SECONDS[(away, lab_away)] += dt
            SECONDS[(home, lab_home)] += dt

        cur = nxt

        # natural penalty ends at time cur
        for p in active:
            if (not p["ended"]) and p["end"] <= cur + 1e-9:
                p["ended"] = True
        active = [p for p in active if not p["ended"]]

        # process all events at time cur
        while ptr < len(events) and abs(events[ptr][1] - cur) < 1e-9:
            etype, t, idx = events[ptr]
            if etype == "P_START":
                active.append(penalties[idx])

            else:  # GOAL
                row = ggame.loc[idx]
                scoring_team = str(row["Team"])
                defending_team = home if scoring_team == away else away

                # state at goal moment (after P_STARTs at same t already applied)
                sA, sH, phase = strengths(active, t, away, home)

                if scoring_team == away:
                    team_skaters, opp_skaters = sA, sH
                else:
                    team_skaters, opp_skaters = sH, sA

                lab_scoring = sit_label(team_skaters, opp_skaters, phase)
                lab_def     = sit_label(opp_skaters, team_skaters, phase)

                GF[(scoring_team, lab_scoring)] += 1
                GA[(defending_team, lab_def)] += 1

                # if goal scored with manpower advantage, end earliest-ending MINOR on defending team
                if team_skaters > opp_skaters:
                    def_pens = [p for p in active if (p["team"] == defending_team and not p["ended"])]
                    def_minors = [p for p in def_pens if not p["is_major"]]
                    if def_minors:
                        end_pen = min(def_minors, key=lambda p: p["end"])
                        end_pen["ended"] = True
                        active = [p for p in active if not p["ended"]]

            ptr += 1

# Run all games
for _, r in game_teams.iterrows():
    simulate_game(r["Game_ID"], str(r["Away_Team"]), str(r["Home_Team"]))

# ============================
# BUILD OUTPUT TABLES
# ============================
# Situations requested (deduped) — we’ll output these (plus anything that occurs, if you want)
requested = [
    "5v5","5v4","5v3","4v4","4v3",
    "3v3 (NON OT)","3v3 (OT)",
    "3v4","3v5","4v5"
]
requested = list(dict.fromkeys(requested))  # preserve order, remove dupes

teams = sorted(set([k[0] for k in SECONDS.keys()] + [k[0] for k in GF.keys()] + [k[0] for k in GA.keys()]))

rows = []
for team in teams:
    for lab in requested:
        rows.append({
            "Team": team,
            "Situation": lab,
            "Goals_For": GF.get((team, lab), 0),
            "Goals_Against": GA.get((team, lab), 0),
            "Instances": INSTANCES.get((team, lab), 0),
            "Time_Seconds": int(round(SECONDS.get((team, lab), 0))),
        })

long_df = pd.DataFrame(rows)

# Wide: columns = each stat for each situation
wide_df = long_df.pivot_table(index="Team",
                              columns="Situation",
                              values=["Goals_For","Goals_Against","Instances","Time_Seconds"],
                              aggfunc="sum",
                              fill_value=0)

# Flatten multiindex columns to nice names
wide_df.columns = [f"{stat}__{sit}" for stat, sit in wide_df.columns]
wide_df = wide_df.reset_index()

# ============================
# SAVE
# ============================
long_out = temp_folder / "pppk_situations_long.csv"
wide_out = temp_folder / "pppk_situations_wide.csv"

long_df.to_csv(long_out, index=False)
wide_df.to_csv(wide_out, index=False)

print("Saved:")
print(" -", long_out, "| rows:", len(long_df), "cols:", long_df.shape[1])
print(" -", wide_out, "| rows:", len(wide_df), "cols:", wide_df.shape[1])


Saved:
 - c:\Users\jbanc\OneDrive\Desktop\Project\college_hockey_2526\TEMP\pppk_situations_long.csv | rows: 680 cols: 6
 - c:\Users\jbanc\OneDrive\Desktop\Project\college_hockey_2526\TEMP\pppk_situations_wide.csv | rows: 68 cols: 41


## MON/TUES

In [17]:
## VERSION 1


## Grab tables as dataframes0 penaltiy_summary and scoring_summary
pen = pd.read_sql("SELECT * FROM penalty_summary;", conn)
goals = pd.read_sql("SELECT * FROM scoring_summary;", conn)

def period_start_seconds(period):
    return {
        "1st Period": 0,
        "2nd Period": 20 * 60,
        "3rd Period": 40 * 60,
        "Overtime": 60 * 60,
    }[period]

def parse_mmss(s):
    m, sec = str(s).strip().split(":")
    return int(m) * 60 + int(sec)

pen["t"] = pen["Time"].apply(parse_mmss) + pen["Period"].apply(period_start_seconds)
goals["t"] = goals["Time"].apply(parse_mmss) + goals["Period"].apply(period_start_seconds)

# Game team mapping (penalty table doesn’t have Home/Away, so we borrow it from scoring)
game_teams = goals.groupby("Game_ID")[["Away_Team", "Home_Team"]].first().reset_index()

REG_BASE = 5
OT_BASE = 3

def situation_key(a, b): return f"{a}v{b}"

def strengths(active, t, away, home):
    # count active manpower penalties (ignore misconducts by filtering earlier)
    cA = sum(1 for p in active if p["team"] == away and not p["ended"])
    cH = sum(1 for p in active if p["team"] == home and not p["ended"])
    phase = "OT" if t >= 3600 else "REG"

    if phase == "REG":
        sA = max(3, REG_BASE - cA)
        sH = max(3, REG_BASE - cH)
    else:
        # 3v3 OT: penalties create 4v3 by ADDING a skater to the non-penalized team
        sA = OT_BASE + max(0, cH - cA)
        sH = OT_BASE + max(0, cA - cH)

    return sA, sH, phase

def simulate_game(game_id, away, home, ot_length=300):
    pgame = pen[(pen["Game_ID"] == game_id) & (pen["Pen_Length"].isin([2, 5]))].copy()
    ggame = goals[goals["Game_ID"] == game_id].copy()

    pgame = pgame.sort_values("t").reset_index(drop=True)
    ggame = ggame.sort_values("t").reset_index(drop=True)

    # events: penalties start + goals
    events = []
    for i, r in pgame.iterrows(): events.append(("P_START", r.t, i))
    for j, r in ggame.iterrows(): events.append(("GOAL", r.t, j))
    events.sort(key=lambda x: (x[1], 0 if x[0] == "P_START" else 1))

    penalties = []
    for i, r in pgame.iterrows():
        penalties.append({
            "team": str(r["Team"]),
            "start": float(r["t"]),
            "length": int(r["Pen_Length"]),
            "is_major": int(r["Pen_Length"]) == 5,
            "is_coincidental": bool(r.get("is_coincidental", False)),
            "end": float(r["t"]) + int(r["Pen_Length"]) * 60.0,
            "ended": False,
        })

    max_t = max([0] + pgame.t.tolist() + ggame.t.tolist())
    end_time = 3600 if max_t <= 3600 else 3600 + ot_length

    active = []
    time_by_team = defaultdict(float)
    gf = defaultdict(int)
    ga = defaultdict(int)
    enriched = []

    score_a = 0
    score_h = 0

    cur = 0
    ptr = 0

    def allocate(t0, t1):
        if t1 <= t0: return
        sA, sH, phase = strengths(active, t0, away, home)
        time_by_team[(away, situation_key(sA, sH), phase)] += (t1 - t0)
        time_by_team[(home, situation_key(sH, sA), phase)] += (t1 - t0)

    while cur < end_time:
        next_event = events[ptr][1] if ptr < len(events) else end_time
        next_end = min([p["end"] for p in active if not p["ended"]] + [end_time])
        nxt = min(next_event, next_end, end_time)

        allocate(cur, nxt)
        cur = nxt

        # natural penalty ends
        for p in active:
            if (not p["ended"]) and p["end"] <= cur + 1e-9:
                p["ended"] = True
        active = [p for p in active if not p["ended"]]

        # process events at this time
        while ptr < len(events) and events[ptr][1] == cur:
            etype, t, idx = events[ptr]

            if etype == "P_START":
                active.append(penalties[idx])

            else:  # GOAL
                row = ggame.loc[idx]
                scoring = row.Team
                defending = home if scoring == away else away

                sA, sH, phase = strengths(active, t, away, home)
                ts = sA if scoring == away else sH
                os = sH if scoring == away else sA

                # score diff before goal (scoring team perspective)
                if scoring == away:
                    diff = score_a - score_h
                    score_a += 1
                else:
                    diff = score_h - score_a
                    score_h += 1

                def_pens = [p for p in active if p["team"] == defending and not p["ended"]]
                def_minors = [p for p in def_pens if not p["is_major"]]
                num_def = len(def_pens)

                oldest = min([p["start"] for p in def_pens], default=np.nan)
                newest = max([p["start"] for p in def_pens], default=np.nan)

                ended_pen = None
                if ts > os and def_minors:
                    ended_pen = min(def_minors, key=lambda p: p["end"])
                    ended_pen["ended"] = True

                enriched.append({
                    "Game_ID": game_id,
                    "t": t,
                    "Period": row.Period,
                    "Time": row.Time,
                    "Scoring_Team": scoring,
                    "Defending_Team": defending,
                    "Team_Skaters": ts,
                    "Opp_Skaters": os,
                    "Situation": situation_key(ts, os),
                    "Phase": phase,
                    "Score_Diff_Before": diff,
                    "Def_Penalties_Active": num_def,
                    "Secs_Since_Oldest_Def_Pen_Start": (t - oldest) if num_def else np.nan,
                    "Secs_Since_Newest_Def_Pen_Start": (t - newest) if num_def else np.nan,
                    "Ended_Penalty": ended_pen is not None,
                    "Secs_Into_Ended_Penalty": (t - ended_pen["start"]) if ended_pen is not None else np.nan,
                })

                gf[(scoring, situation_key(ts, os), phase)] += 1
                ga[(defending, situation_key(os, ts), phase)] += 1

                active = [p for p in active if not p.get("ended", False)]

            ptr += 1

    return time_by_team, gf, ga, pd.DataFrame(enriched)

# --- Run all games ---
all_time = defaultdict(float)
all_gf = defaultdict(int)
all_ga = defaultdict(int)
enriched_all = []

for _, r in game_teams.iterrows():
    tb, gf, ga, eg = simulate_game(r.Game_ID, r.Away_Team, r.Home_Team)
    for k, v in tb.items(): all_time[k] += v
    for k, v in gf.items(): all_gf[k] += v
    for k, v in ga.items(): all_ga[k] += v
    enriched_all.append(eg)

enriched_goals = pd.concat(enriched_all, ignore_index=True)

# Long summary
time_rows = [{"Team":k[0], "Situation":k[1], "Phase":k[2], "Seconds":v} for k, v in all_time.items()]
gf_rows   = [{"Team":k[0], "Situation":k[1], "Phase":k[2], "GF":v} for k, v in all_gf.items()]
ga_rows   = [{"Team":k[0], "Situation":k[1], "Phase":k[2], "GA":v} for k, v in all_ga.items()]

summary = (pd.DataFrame(time_rows)
           .merge(pd.DataFrame(gf_rows), on=["Team","Situation","Phase"], how="left")
           .merge(pd.DataFrame(ga_rows), on=["Team","Situation","Phase"], how="left")
           .fillna({"GF":0,"GA":0}))
summary["GF"] = summary["GF"].astype(int)
summary["GA"] = summary["GA"].astype(int)

# Save
summary.to_csv(temp_folder / "manpower_time_and_goals_long.csv", index=False)
enriched_goals.to_csv(temp_folder / "pp_goal_game_state_enriched.csv", index=False)


## Long Summary Table

In [18]:
## ========= CREATES A LONG TABLE WITH MANPOWER SITUATION, TIME, AND GOALS FOR PP/PK/EV =========


def parse_sit(s):
    a,b = s.split("v")
    return int(a), int(b)

ts_os = summary["Situation"].apply(parse_sit)
summary["Team_Skaters"] = ts_os.apply(lambda x: x[0])
summary["Opp_Skaters"]  = ts_os.apply(lambda x: x[1])
summary["Advantage"] = summary["Team_Skaters"] - summary["Opp_Skaters"]
summary["Role"] = np.select(
    [summary["Advantage"]>0, summary["Advantage"]<0],
    ["PP","PK"],
    default="EV"
)

# Per-60 rates (avoid div by 0)
summary["Min"] = summary["Seconds"] / 60.0
summary["GF_per60"] = np.where(summary["Seconds"]>0, summary["GF"] * 3600.0 / summary["Seconds"], np.nan)
summary["GA_per60"] = np.where(summary["Seconds"]>0, summary["GA"] * 3600.0 / summary["Seconds"], np.nan)
summary["NetGF"] = summary["GF"] - summary["GA"]
summary["NetGF_per60"] = np.where(summary["Seconds"]>0, summary["NetGF"] * 3600.0 / summary["Seconds"], np.nan)

# A nicer sort key: PP first, EV, PK; and within role, bigger advantage first
role_order = {"PP":0,"EV":1,"PK":2}
summary["Role_Order"] = summary["Role"].map(role_order)
summary["Abs_Adv"] = summary["Advantage"].abs()

combined = summary.sort_values(["Team","Phase","Role_Order","Advantage"], ascending=[True, True, True, False]).copy()
combined.drop(columns=["Role_Order","Abs_Adv"], inplace=True)

out_path = Path(temp_folder / "manpower_time_and_goals_long_with_pp_pk.csv")
combined.to_csv(out_path, index=False)




In [19]:

pen = pd.read_sql("SELECT * FROM penalty_summary;", conn)
goals = pd.read_sql("SELECT * FROM scoring_summary;", conn)

# --- helpers ---
def period_start_seconds(period: str) -> int:
    m = {
        "1st Period": 0,
        "2nd Period": 20 * 60,
        "3rd Period": 40 * 60,
        "Overtime": 60 * 60,
        "OT": 60 * 60,
    }
    return m.get(str(period).strip(), 0)

def parse_mmss(s: str) -> int:
    s = str(s).strip()
    if ":" not in s:
        # already seconds or empty
        try:
            return int(float(s))
        except Exception:
            return 0
    mm, ss = s.split(":")
    return int(mm) * 60 + int(ss)

# absolute seconds from game start
pen["t"] = pen["Time"].apply(parse_mmss) + pen["Period"].apply(period_start_seconds)
goals["t"] = goals["Time"].apply(parse_mmss) + goals["Period"].apply(period_start_seconds)

# Normalize basic columns (best-effort)
for col in ["Team", "Player", "Penalty_Type"]:
    if col in pen.columns:
        pen[col] = pen[col].astype(str)

# Filter to manpower penalties: keep common lengths, drop 0/NaN
pen["Pen_Length"] = pd.to_numeric(pen["Pen_Length"], errors="coerce")
pen = pen[pen["Pen_Length"].notna()].copy()
pen["Pen_Length"] = pen["Pen_Length"].astype(int)

# Expand 4-minute double minors into 2 + 2 (approximation; goal ends first half)
expanded_rows = []
for _, r in pen.iterrows():
    L = int(r.Pen_Length)
    if L == 4:
        r1 = r.copy()
        r2 = r.copy()
        r1["Pen_Length"] = 2
        r2["Pen_Length"] = 2
        r2["t"] = r["t"] + 120  # second half starts after 2 min
        expanded_rows.append(r1)
        expanded_rows.append(r2)
    else:
        expanded_rows.append(r)
pen = pd.DataFrame(expanded_rows).reset_index(drop=True)

# Keep only typical manpower lengths
pen = pen[pen["Pen_Length"].isin([2, 5])].copy()

# Game team mapping from scoring (preferred); fallback to penalty table teams if needed
if {"Away_Team","Home_Team"}.issubset(goals.columns):
    game_teams = goals.groupby("Game_ID")[["Away_Team","Home_Team"]].first().reset_index()
else:
    # fallback: infer two teams per game from penalty table
    tmp = (pen.groupby("Game_ID")["Team"].unique().reset_index())
    rows = []
    for _, r in tmp.iterrows():
        teams = list(r["Team"])
        if len(teams) >= 2:
            rows.append({"Game_ID": r["Game_ID"], "Away_Team": teams[0], "Home_Team": teams[1]})
    game_teams = pd.DataFrame(rows)

REG_BASE = 5
OT_BASE = 3

def strengths(active, t, away, home):
    # count active penalties by team (manpower only)
    cA = sum(1 for p in active if p["team"] == away and not p["ended"])
    cH = sum(1 for p in active if p["team"] == home and not p["ended"])
    phase = "OT" if t >= 3600 else "REG"
    if phase == "REG":
        sA = max(3, REG_BASE - cA)
        sH = max(3, REG_BASE - cH)
    else:
        # 3v3 OT: penalties create 4v3 by adding a skater to non-penalized team
        sA = OT_BASE + max(0, cH - cA)
        sH = OT_BASE + max(0, cA - cH)
    return sA, sH, phase

def adv_state(sA, sH, away, home):
    if sA > sH:
        return away, home, sA, sH, sA - sH
    if sH > sA:
        return home, away, sH, sA, sH - sA
    return None, None, sA, sH, 0

def simulate_windows_for_game(game_id, away, home, ot_length=300):
    pgame = pen[pen["Game_ID"] == game_id].copy().sort_values("t").reset_index(drop=True)
    ggame = goals[goals["Game_ID"] == game_id].copy().sort_values("t").reset_index(drop=True)

    # penalties as objects
    penalties = []
    for i, r in pgame.iterrows():
        penalties.append({
            "team": r["Team"],
            "start": float(r["t"]),
            "length": int(r["Pen_Length"]),
            "is_major": int(r["Pen_Length"]) == 5,
            "end": float(r["t"]) + int(r["Pen_Length"]) * 60,
            "ended": False,
        })

    events = []
    for i, r in pgame.iterrows(): events.append(("P_START", float(r["t"]), i))
    for j, r in ggame.iterrows(): events.append(("GOAL", float(r["t"]), j))
    events.sort(key=lambda x: (x[1], 0 if x[0] == "P_START" else 1))

    max_t = max([0.0] + pgame["t"].tolist() + ggame["t"].tolist())
    end_time = 3600.0 if max_t <= 3600.0 else 3600.0 + float(ot_length)

    active = []
    ptr = 0
    cur = 0.0

    score_a = 0
    score_h = 0

    # current open PP window (only one team can be advantaged at a time in this model)
    win = None
    win_rows = []

    # Track per-window situation seconds (from PP team perspective)
    # We'll store into win["sit_secs"][sit_str] during allocation
    def open_window(t, pp_team, pk_team, team_skaters, opp_skaters, phase):
        nonlocal win, score_a, score_h
        # score diff from pp_team perspective at start
        if pp_team == away:
            diff = score_a - score_h
        else:
            diff = score_h - score_a
        win = {
            "Game_ID": game_id,
            "PP_Team": pp_team,
            "PK_Team": pk_team,
            "Phase": phase,
            "Start_t": t,
            "End_t": np.nan,
            "Start_Period": None,
            "End_Period": None,
            "Start_ScoreDiff_PP": diff,
            "End_ScoreDiff_PP": np.nan,
            "PP_Goals": 0,
            "SH_Goals": 0,   # goals by PK team during this window
            "Ended_By": None,
            "sit_secs": defaultdict(float),
            "sit_gf": defaultdict(int),  # PP goals by situation
            "sit_ga": defaultdict(int),  # SH goals against by situation
        }

    def close_window(t, reason):
        nonlocal win, score_a, score_h
        if win is None:
            return
        win["End_t"] = t
        # end score diff from PP perspective AFTER events at time t are processed already in loop;
        # for close, use current score
        pp_team = win["PP_Team"]
        if pp_team == away:
            diff = score_a - score_h
        else:
            diff = score_h - score_a
        win["End_ScoreDiff_PP"] = diff
        win["Ended_By"] = reason
        # flatten sit_secs to columns later; keep dict for now
        win_rows.append(win)
        win = None

    def situation_str(ts, os):
        return f"{int(ts)}v{int(os)}"

    def allocate_segment(t0, t1):
        nonlocal win
        if t1 <= t0:
            return
        sA, sH, phase = strengths(active, t0, away, home)
        pp_team, pk_team, ts, os, adv = adv_state(sA, sH, away, home)
        if pp_team is None or adv <= 0:
            return
        # ensure window open and matches
        if win is None or win["PP_Team"] != pp_team:
            # shouldn't happen if transitions handled, but guard
            open_window(t0, pp_team, pk_team, ts, os, phase)
        sit = situation_str(ts, os)
        win["sit_secs"][sit] += (t1 - t0)

    # Initialize window state at t=0
    sA0, sH0, phase0 = strengths(active, 0.0, away, home)
    pp0, pk0, ts0, os0, adv0 = adv_state(sA0, sH0, away, home)
    if pp0 is not None and adv0 > 0:
        open_window(0.0, pp0, pk0, ts0, os0, phase0)

    # Main loop
    while cur < end_time - 1e-9:
        next_event_t = events[ptr][1] if ptr < len(events) else end_time
        next_end_t = min([p["end"] for p in active if not p["ended"]] + [end_time])
        nxt = min(next_event_t, next_end_t, end_time)

        # allocate segment with current state
        allocate_segment(cur, nxt)
        cur = nxt

        # penalty natural ends at this time
        for p in active:
            if (not p["ended"]) and p["end"] <= cur + 1e-9:
                p["ended"] = True
        # remove ended
        active = [p for p in active if not p["ended"]]

        # process all events at cur
        reason_trigger = None
        while ptr < len(events) and abs(events[ptr][1] - cur) < 1e-9:
            etype, t, idx = events[ptr]
            if etype == "P_START":
                active.append(penalties[idx])
                reason_trigger = "new_penalty"
            else:
                row = ggame.loc[idx]
                scoring = row["Team"]
                defending = home if scoring == away else away

                # state at goal time (after any P_START at same timestamp already processed due to sorting)
                sA, sH, phase = strengths(active, t, away, home)
                pp_team, pk_team, ts, os, adv = adv_state(sA, sH, away, home)

                # Update score diff before goal isn't needed here; just update score
                if scoring == away:
                    score_a += 1
                else:
                    score_h += 1

                # If there is an open window, attribute goals if it happens within an advantage state
                if pp_team is not None and adv > 0:
                    sit = situation_str(ts, os)
                    # PP goal
                    if scoring == pp_team:
                        if win is not None and win["PP_Team"] == pp_team:
                            win["PP_Goals"] += 1
                            win["sit_gf"][sit] += 1
                        # end a minor on defending team if any
                        def_pens = [p for p in active if p["team"] == defending and not p["ended"]]
                        def_minors = [p for p in def_pens if (not p["is_major"] and not p.get("is_coincidental", False))]

                        if def_minors:
                            end_pen = min(def_minors, key=lambda p: p["end"])
                            end_pen["ended"] = True
                            reason_trigger = "pp_goal_ended_minor"
                            # remove ended immediately
                            active = [p for p in active if not p["ended"]]
                        else:
                            reason_trigger = "pp_goal_no_minor_to_end"
                    else:
                        # shorthand goal by PK team
                        if win is not None and win["PP_Team"] == pp_team:
                            win["SH_Goals"] += 1
                            win["sit_ga"][sit] += 1
                        reason_trigger = "sh_goal"
                else:
                    reason_trigger = "ev_goal"
            ptr += 1

        # after processing ends/events at cur, determine new advantage state
        sA2, sH2, phase2 = strengths(active, cur, away, home)
        pp2, pk2, ts2, os2, adv2 = adv_state(sA2, sH2, away, home)
        current_pp = win["PP_Team"] if win is not None else None

        if current_pp is not None:
            # If advantage ended or switched teams, close
            if pp2 is None or pp2 != current_pp:
                close_window(cur, reason_trigger or "penalty_end_or_state_change")

        # If new advantage starts (and no open window), open
        if win is None and pp2 is not None and adv2 > 0:
            open_window(cur, pp2, pk2, ts2, os2, phase2)

    # close any remaining
    if win is not None:
        close_window(end_time, "end_of_game")

    return win_rows

# Run all games, build windows dataframe
all_windows = []
for _, r in game_teams.iterrows():
    game_id = r["Game_ID"]
    away = r["Away_Team"]
    home = r["Home_Team"]
    all_windows.extend(simulate_windows_for_game(game_id, away, home))

windows_df = pd.DataFrame(all_windows)

# Flatten situation seconds/goals dicts into long tables first (cleaner for analytics)
# window_id
windows_df = windows_df.reset_index(drop=True)
windows_df["Window_ID"] = windows_df.index.astype(int)

# Duration
windows_df["Duration_Seconds"] = windows_df["End_t"] - windows_df["Start_t"]

# Build long situation breakdowns
sit_rows = []
for _, w in windows_df.iterrows():
    wid = int(w["Window_ID"])
    for sit, sec in (w["sit_secs"] or {}).items():
        sit_rows.append({
            "Window_ID": wid,
            "Game_ID": w["Game_ID"],
            "PP_Team": w["PP_Team"],
            "PK_Team": w["PK_Team"],
            "Phase": w["Phase"],
            "Situation_PP": sit,
            "Seconds": sec,
            "PP_GF": int((w["sit_gf"] or {}).get(sit, 0)),
            "PP_GA": int((w["sit_ga"] or {}).get(sit, 0)),  # SH goals against during this sit
        })
sit_long = pd.DataFrame(sit_rows)

# Clean dict columns out of the main windows table for CSV
windows_out = windows_df.drop(columns=["sit_secs","sit_gf","sit_ga"])

# ----- Build the "ultimate" wide table -----
# Start from your existing manpower long (with PP/PK) if present
# mp_long_path = Path(temp_folder / "manpower_time_and_goals_long_with_pp_pk.csv")
if mp_long_path.exists():
    mp_long = pd.read_csv(mp_long_path)
else:
    mp_long = pd.read_csv(Path(temp_folder / "manpower_time_and_goals_long.csv"))
    # derive Role quickly
    def parse_sit(s):
        a,b = s.split("v"); return int(a), int(b)
    tsos = mp_long["Situation"].apply(parse_sit)
    mp_long["Team_Skaters"] = tsos.apply(lambda x: x[0])
    mp_long["Opp_Skaters"] = tsos.apply(lambda x: x[1])
    mp_long["Advantage"] = mp_long["Team_Skaters"] - mp_long["Opp_Skaters"]
    mp_long["Role"] = np.select([mp_long["Advantage"]>0, mp_long["Advantage"]<0], ["PP","PK"], default="EV")

# Wide manpower by key situations (REG only default; we'll do both phases separately)
def make_wide_manpower(df, phase=None):
    d = df.copy()
    if phase is not None:
        d = d[d["Phase"]==phase].copy()
        suffix = f"_{phase}"
    else:
        suffix = ""
    # only PP/PK relevant roles
    d = d[d["Role"].isin(["PP","PK"])].copy()
    # build column labels from Role + Situation (team perspective)
    d["Col"] = d["Role"] + "_" + d["Situation"]
    secs = d.pivot_table(index="Team", columns="Col", values="Seconds", aggfunc="sum", fill_value=0)
    gf   = d.pivot_table(index="Team", columns="Col", values="GF", aggfunc="sum", fill_value=0)
    ga   = d.pivot_table(index="Team", columns="Col", values="GA", aggfunc="sum", fill_value=0)
    secs.columns = [f"Seconds{suffix}_{c}" for c in secs.columns]
    gf.columns   = [f"GF{suffix}_{c}" for c in gf.columns]
    ga.columns   = [f"GA{suffix}_{c}" for c in ga.columns]
    out = secs.join(gf).join(ga)
    return out.reset_index()

wide_mp_all = make_wide_manpower(mp_long, phase=None)

# Window-based aggregates (these define "opportunities")
pp_agg = (windows_out.groupby("PP_Team", as_index=False)
          .agg(PP_Opps=("Window_ID","count"),
               PP_Seconds=("Duration_Seconds","sum"),
               PP_Goals=("PP_Goals","sum"),
               SHGA=("SH_Goals","sum")))
pp_agg = pp_agg.rename(columns={"PP_Team":"Team"})

pk_agg = (windows_out.groupby("PK_Team", as_index=False)
          .agg(PK_Opps=("Window_ID","count"),
               PK_Seconds=("Duration_Seconds","sum"),
               PK_GA=("PP_Goals","sum"),   # goals allowed while shorthanded
               SHGF=("SH_Goals","sum")))   # shorties scored
pk_agg = pk_agg.rename(columns={"PK_Team":"Team"})

wide = wide_mp_all.merge(pp_agg, on="Team", how="left").merge(pk_agg, on="Team", how="left")
for c in ["PP_Opps","PP_Seconds","PP_Goals","SHGA","PK_Opps","PK_Seconds","PK_GA","SHGF"]:
    if c in wide.columns:
        wide[c] = wide[c].fillna(0)
        if c.endswith("Opps") or c in ["PP_Goals","SHGA","PK_GA","SHGF"]:
            wide[c] = wide[c].astype(int)
## ===== PATCH
# --- after you create wide_mp_all, pp_agg, pk_agg ---

# Make a master list of teams from ALL sources (so base can’t be empty)
teams = set()

if "Team" in wide_mp_all.columns and len(wide_mp_all) > 0:
    teams |= set(wide_mp_all["Team"].dropna().unique())

if "Team" in pp_agg.columns and len(pp_agg) > 0:
    teams |= set(pp_agg["Team"].dropna().unique())

if "Team" in pk_agg.columns and len(pk_agg) > 0:
    teams |= set(pk_agg["Team"].dropna().unique())

# As a fallback, pull from goals if needed
if len(teams) == 0 and "Team" in goals.columns:
    teams |= set(goals["Team"].dropna().unique())

teams = sorted(teams)

# Build wide table from master team frame
wide = (pd.DataFrame({"Team": teams})
        .merge(wide_mp_all, on="Team", how="left")
        .merge(pp_agg, on="Team", how="left")
        .merge(pk_agg, on="Team", how="left"))

# Fill numeric NaNs
for c in wide.columns:
    if c != "Team":
        wide[c] = wide[c].fillna(0)

# Re-cast the opp/goal count columns (optional but nice)
count_cols = ["PP_Opps","PP_Goals","SHGA","PK_Opps","PK_GA","SHGF"]
for c in count_cols:
    if c in wide.columns:
        wide[c] = wide[c].astype(int)



# Derived metrics
wide["PP_Min"] = wide["PP_Seconds"] / 60.0
wide["PK_Min"] = wide["PK_Seconds"] / 60.0
wide["PP%"] = np.where(wide["PP_Opps"]>0, wide["PP_Goals"]/wide["PP_Opps"], np.nan)
wide["PK%"] = np.where(wide["PK_Opps"]>0, 1.0 - (wide["PK_GA"]/wide["PK_Opps"]), np.nan)

wide["PP_GF60"] = np.where(wide["PP_Seconds"]>0, wide["PP_Goals"]*3600.0/wide["PP_Seconds"], np.nan)
wide["PP_GA60"] = np.where(wide["PP_Seconds"]>0, wide["SHGA"]*3600.0/wide["PP_Seconds"], np.nan)
wide["PP_NetGF60"] = np.where(wide["PP_Seconds"]>0, (wide["PP_Goals"]-wide["SHGA"])*3600.0/wide["PP_Seconds"], np.nan)

wide["PK_GF60"] = np.where(wide["PK_Seconds"]>0, wide["SHGF"]*3600.0/wide["PK_Seconds"], np.nan)
wide["PK_GA60"] = np.where(wide["PK_Seconds"]>0, wide["PK_GA"]*3600.0/wide["PK_Seconds"], np.nan)
wide["PK_NetGF60"] = np.where(wide["PK_Seconds"]>0, (wide["SHGF"]-wide["PK_GA"])*3600.0/wide["PK_Seconds"], np.nan)

# Save outputs
out_dir = temp_folder
windows_csv = out_dir / "pp_opportunity_windows.csv"
sit_long_csv = out_dir / "pp_opportunity_window_situations_long.csv"
wide_csv = out_dir / "ultimate_pp_pk_analytics_wide.csv"

# windows_out.to_csv(windows_csv, index=False) # optional to save the main windows table
# sit_long.to_csv(sit_long_csv, index=False) # optional to save the long situation breakdowns
wide.to_csv(wide_csv, index=False) #


NameError: name 'mp_long_path' is not defined


## Create Ultimate wide table

In [ ]:
mp_long["Situation"] = (mp_long["Situation"].astype(str)
                        .str.replace(" ", "", regex=False)
                        .str.replace("on", "v", regex=False)
                        .str.replace("-", "v", regex=False))


In [ ]:
import numpy as np
import pandas as pd

# -----------------------------
# ULTIMATE WIDE TABLE (PP+PK)
# Requires: mp_long, windows_out
# -----------------------------

# 1) Normalize Situation to "5v4" style
# mp_long = 
mp_long["Situation"] = (
    mp_long["Situation"].astype(str)
    .str.strip()
    .str.replace(" ", "", regex=False)
    .str.replace("on", "v", regex=False)
    .str.replace("-", "v", regex=False)
)

# 2) Derive Role if missing (PP / PK / EV)
if "Role" not in mp_long.columns:
    def _parse_sit(x):
        try:
            a, b = str(x).split("v")
            return int(a), int(b)
        except Exception:
            return np.nan, np.nan

    tsos = mp_long["Situation"].apply(_parse_sit)
    mp_long["Team_Skaters"] = tsos.apply(lambda t: t[0])
    mp_long["Opp_Skaters"]  = tsos.apply(lambda t: t[1])
    mp_long["Advantage"] = mp_long["Team_Skaters"] - mp_long["Opp_Skaters"]
    mp_long["Role"] = np.select(
        [mp_long["Advantage"] > 0, mp_long["Advantage"] < 0],
        ["PP", "PK"],
        default="EV"
    )

# 3) Wide manpower pivot (PP/PK only)
mp_pppk = mp_long[mp_long["Role"].isin(["PP", "PK"])].copy()
mp_pppk["Col"] = mp_pppk["Role"] + "_" + mp_pppk["Situation"]

wide_secs = mp_pppk.pivot_table(index="Team", columns="Col", values="Seconds", aggfunc="sum", fill_value=0)
wide_gf   = mp_pppk.pivot_table(index="Team", columns="Col", values="GF",      aggfunc="sum", fill_value=0)
wide_ga   = mp_pppk.pivot_table(index="Team", columns="Col", values="GA",      aggfunc="sum", fill_value=0)

wide_secs.columns = [f"Seconds_{c}" for c in wide_secs.columns]
wide_gf.columns   = [f"GF_{c}"      for c in wide_gf.columns]
wide_ga.columns   = [f"GA_{c}"      for c in wide_ga.columns]

wide_mp_all = wide_secs.join(wide_gf).join(wide_ga).reset_index()

# 4) Window-based aggregates = opportunities (true “ultimate” layer)
pp_agg = (windows_out.groupby("PP_Team", as_index=False)
          .agg(PP_Opps=("Window_ID", "count"),
               PP_Seconds=("Duration_Seconds", "sum"),
               PP_Goals=("PP_Goals", "sum"),
               SHGA=("SH_Goals", "sum"))
          .rename(columns={"PP_Team": "Team"}))

pk_agg = (windows_out.groupby("PK_Team", as_index=False)
          .agg(PK_Opps=("Window_ID", "count"),
               PK_Seconds=("Duration_Seconds", "sum"),
               PK_GA=("PP_Goals", "sum"),
               SHGF=("SH_Goals", "sum"))
          .rename(columns={"PK_Team": "Team"}))

# 5) MASTER TEAM LIST (prevents empty ultimate table)
teams = sorted(set(wide_mp_all["Team"].dropna().unique())
               | set(pp_agg["Team"].dropna().unique())
               | set(pk_agg["Team"].dropna().unique()))

wide = (pd.DataFrame({"Team": teams})
        .merge(wide_mp_all, on="Team", how="left")
        .merge(pp_agg, on="Team", how="left")
        .merge(pk_agg, on="Team", how="left"))

# 6) Fill and derived metrics
for c in wide.columns:
    if c != "Team":
        wide[c] = wide[c].fillna(0)

# cast key count cols back to int
for c in ["PP_Opps","PP_Goals","SHGA","PK_Opps","PK_GA","SHGF"]:
    if c in wide.columns:
        wide[c] = wide[c].astype(int)

wide["PP%"] = np.where(wide["PP_Opps"] > 0, wide["PP_Goals"] / wide["PP_Opps"], np.nan)
wide["PK%"] = np.where(wide["PK_Opps"] > 0, 1.0 - (wide["PK_GA"] / wide["PK_Opps"]), np.nan)

wide["PP_GF60"] = np.where(wide["PP_Seconds"] > 0, wide["PP_Goals"] * 3600.0 / wide["PP_Seconds"], np.nan)
wide["PP_GA60"] = np.where(wide["PP_Seconds"] > 0, wide["SHGA"]     * 3600.0 / wide["PP_Seconds"], np.nan)
wide["PP_NetGF60"] = np.where(wide["PP_Seconds"] > 0, (wide["PP_Goals"] - wide["SHGA"]) * 3600.0 / wide["PP_Seconds"], np.nan)

wide["PK_GF60"] = np.where(wide["PK_Seconds"] > 0, wide["SHGF"] * 3600.0 / wide["PK_Seconds"], np.nan)
wide["PK_GA60"] = np.where(wide["PK_Seconds"] > 0, wide["PK_GA"] * 3600.0 / wide["PK_Seconds"], np.nan)
wide["PK_NetGF60"] = np.where(wide["PK_Seconds"] > 0, (wide["SHGF"] - wide["PK_GA"]) * 3600.0 / wide["PK_Seconds"], np.nan)

# 7) Save
wide.to_csv(temp_folder / "ultimate_pp_pk_analytics_wide.csv", index=False)
print("Saved:", "ultimate_pp_pk_analytics_wide.csv", "| rows:", len(wide), "| cols:", wide.shape[1])


Saved: ultimate_pp_pk_analytics_wide.csv | rows: 63 | cols: 17
